# 🎯 Sentiment Analysis using NLP Pipeline & ML Models
### Data Science Internship – February 2026
**Task:** Build an end-to-end Sentiment Analysis system using NLP preprocessing, feature engineering, and multiple ML models.

---
**Pipeline Flow:**
```
Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation → Comparison
```

## ⚙️ Step 0: Install & Import Libraries

In [ ]:
# ── Install required libraries (uncomment if running on Google Colab) ──
# !pip install nltk scikit-learn pandas numpy matplotlib seaborn --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

# ── NLP Libraries ──
import nltk
nltk.download('stopwords',    quiet=True)
nltk.download('punkt',        quiet=True)
nltk.download('wordnet',      quiet=True)
nltk.download('omw-1.4',      quiet=True)
from nltk.corpus   import stopwords
from nltk.stem     import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ── Feature Engineering ──
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# ── ML Models ──
from sklearn.linear_model    import LogisticRegression
from sklearn.naive_bayes     import MultinomialNB
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier

# ── Evaluation ──
from sklearn.model_selection  import train_test_split
from sklearn.metrics          import (accuracy_score, precision_score,
                                      recall_score, f1_score,
                                      classification_report,
                                      confusion_matrix, ConfusionMatrixDisplay)

print('✅ All libraries imported successfully!')

---
## 📦 Step 1: Data Loading & Understanding

> **Dataset:** We create a realistic IMDb-style dataset with 1 000 labelled reviews.  
> To use a real Kaggle dataset instead, replace the cell below with:  
> `df = pd.read_csv('IMDB Dataset.csv')` after downloading from Kaggle.

In [ ]:
# ── Build a realistic synthetic IMDb-style dataset ──
import random
random.seed(42)
np.random.seed(42)

positive_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the story was gripping.",
    "One of the best films I have ever seen. Highly recommend to everyone!",
    "Amazing cinematography and a brilliant soundtrack. A true masterpiece.",
    "The plot was engaging from start to finish. The characters felt very real.",
    "Incredible performances by all the cast members. I was moved to tears.",
    "A heartwarming story that left me smiling for days. Truly wonderful!",
    "Brilliant direction and outstanding screenplay. Loved every minute of it.",
    "This film exceeded all my expectations. The ending was perfect.",
    "Visually stunning and emotionally powerful. A must watch for all movie lovers.",
    "The best movie I have watched this year. Cannot wait to see it again!",
    "Superb acting and a touching story. This movie deserves all the awards.",
    "What a delightful film! The humor was spot on and the story was sweet.",
    "I was completely hooked from the very first scene. Fantastic filmmaking.",
    "A perfect blend of action and emotion. Loved every scene of this movie.",
    "The director did an amazing job bringing this story to life. Truly remarkable.",
    "Outstanding performance by the lead actor. The chemistry was electric.",
    "A beautifully crafted film with a powerful message. Highly recommended.",
    "This movie restored my faith in modern cinema. Absolutely loved it!",
    "Witty, emotional and brilliantly paced. One of the best I have seen.",
    "The script was sharp and the direction was flawless. A cinematic gem.",
]

negative_reviews = [
    "This was a complete waste of time and money. Terrible in every way.",
    "The worst movie I have ever seen. The plot made absolutely no sense.",
    "Boring, predictable and poorly acted. I fell asleep halfway through.",
    "Awful script and terrible direction. A total disappointment.",
    "I cannot believe how bad this movie was. Do not waste your time on it.",
    "The acting was wooden and the story was full of plot holes.",
    "Extremely disappointing. The trailer was much better than the actual film.",
    "A dull and lifeless film that had no idea where it was going.",
    "Poorly written and poorly directed. One of the worst films of the year.",
    "The characters were one-dimensional and the dialogue was cringe-worthy.",
    "Nothing about this film worked. It was tedious from beginning to end.",
    "An absolute disaster of a movie. The studio should be embarrassed.",
    "Painfully slow and utterly boring. I regret spending money on this.",
    "The special effects were laughable and the story was nonsensical.",
    "Possibly the most boring film I have ever had to sit through.",
    "Terrible pacing and zero character development. A complete mess.",
    "I wanted to walk out after the first fifteen minutes. Dreadful.",
    "The plot was a total shambles and the acting was embarrassing.",
    "Nothing redeemable about this film whatsoever. Save your money.",
    "A hollow and unoriginal film that insults the audience's intelligence.",
]

# ── Generate 1000 reviews by sampling with variation ──
all_reviews = []
all_labels  = []

noise_words = ['!!!', 'Visit http://fakeurl.com', '😍', '😡', '100%', '@user', '#movie']

for _ in range(500):
    r = random.choice(positive_reviews)
    if random.random() > 0.7:                       # add some noise
        r = r + ' ' + random.choice(noise_words)
    all_reviews.append(r)
    all_labels.append('positive')

for _ in range(500):
    r = random.choice(negative_reviews)
    if random.random() > 0.7:
        r = r + ' ' + random.choice(noise_words)
    all_reviews.append(r)
    all_labels.append('negative')

df = pd.DataFrame({'review': all_reviews, 'sentiment': all_labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print('✅ Dataset created successfully!')
print(f'   Shape : {df.shape}')
print(f'   Columns: {list(df.columns)}')
print()
print(df.head(5))

In [ ]:
# ── 1.1 Basic Dataset Info ──
print('=' * 55)
print('DATASET OVERVIEW')
print('=' * 55)
print(f'Total Samples  : {len(df)}')
print(f'Missing Values : {df.isnull().sum().sum()}')
print()

print('Class Distribution:')
print(df['sentiment'].value_counts())
print()

print('Review Length Stats (characters):')
df['review_length'] = df['review'].apply(len)
print(df['review_length'].describe().round(2))

In [ ]:
# ── 1.2 Visualise Class Distribution & Review Lengths ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution bar chart
counts = df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=['#4CAF50', '#F44336'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Review length histogram
for label, color in [('positive', '#4CAF50'), ('negative', '#F44336')]:
    axes[1].hist(df[df['sentiment'] == label]['review_length'],
                 bins=30, alpha=0.6, color=color, label=label)
axes[1].set_title('Review Length Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()
print('📊 Visualisations complete.')

---
## 🧹 Step 2: NLP Preprocessing Pipeline

We build **reusable, modular functions** for every preprocessing step.

In [ ]:
# ── Initialise NLP tools ──
STOPWORDS   = set(stopwords.words('english'))
# Preserve negation words – critical for sentiment
NEGATIONS   = {'no', 'not', 'nor', 'never', 'neither', 'none', 'nothing', "n't"}
STOPWORDS  -= NEGATIONS

stemmer     = PorterStemmer()
lemmatizer  = WordNetLemmatizer()

# ────────────────────────────────────────────
# PREPROCESSING FUNCTIONS
# ────────────────────────────────────────────

def remove_urls(text):
    """Remove http/https URLs and www links."""
    return re.sub(r'http\S+|www\.\S+', '', text)

def remove_emails(text):
    """Remove email-like patterns."""
    return re.sub(r'\S+@\S+', '', text)

def remove_emojis(text):
    """Strip non-ASCII characters (emojis, symbols)."""
    return text.encode('ascii', 'ignore').decode('ascii')

def to_lowercase(text):
    """Convert all text to lowercase."""
    return text.lower()

def remove_punctuation(text):
    """Remove all punctuation marks."""
    return re.sub(r'[^\w\s]', '', text)

def remove_numbers(text):
    """Remove standalone numbers."""
    return re.sub(r'\b\d+\b', '', text)

def remove_extra_spaces(text):
    """Collapse multiple whitespace into a single space."""
    return re.sub(r'\s+', ' ', text).strip()

def handle_repeated_chars(text):
    """Collapse repeated characters: 'sooooo' → 'so'."""
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def tokenize(text):
    """Split text into word tokens."""
    return word_tokenize(text)

def remove_stopwords(tokens):
    """Remove stopwords, preserving negation words."""
    return [t for t in tokens if t not in STOPWORDS]

def remove_short_tokens(tokens, min_len=2):
    """Remove tokens shorter than min_len (keep 'no', 'not')."""
    keep = {'no', 'not'}
    return [t for t in tokens if len(t) >= min_len or t in keep]

def stem_tokens(tokens):
    """Apply Porter Stemming to each token."""
    return [stemmer.stem(t) for t in tokens]

def lemmatize_tokens(tokens):
    """Apply WordNet Lemmatization to each token."""
    return [lemmatizer.lemmatize(t, pos='v') for t in tokens]

# ────────────────────────────────────────────
# FULL PREPROCESSING PIPELINE
# ────────────────────────────────────────────

def preprocess(text, use_stemming=False):
    """
    Full NLP preprocessing pipeline.
    Args:
        text         : raw input string
        use_stemming : True = stemming, False = lemmatization
    Returns:
        Clean string ready for vectorization.
    """
    if not isinstance(text, str) or not text.strip():
        return ''

    # --- Cleaning ---
    text = remove_urls(text)
    text = remove_emails(text)
    text = remove_emojis(text)
    text = handle_repeated_chars(text)
    text = to_lowercase(text)
    text = remove_punctuation(text)
    text = remove_numbers(text)
    text = remove_extra_spaces(text)

    # --- Tokenization ---
    tokens = tokenize(text)

    # --- Stopword Removal ---
    tokens = remove_stopwords(tokens)

    # --- Short Token Removal ---
    tokens = remove_short_tokens(tokens)

    # --- Stemming OR Lemmatization ---
    if use_stemming:
        tokens = stem_tokens(tokens)
    else:
        tokens = lemmatize_tokens(tokens)

    return ' '.join(tokens)


print('✅ All preprocessing functions defined!')

# ── Quick sanity check ──
sample = "This movie was SOOO amazing!!! Visit http://example.com 😍 not bad at all!"
print(f'\nOriginal : {sample}')
print(f'Processed: {preprocess(sample)}')

In [ ]:
# ── Apply preprocessing to entire dataset ──
df['clean_review'] = df['review'].apply(preprocess)

print('✅ Preprocessing applied to all reviews!')
print(f'\nSample comparison:')
for i in range(3):
    print(f'\n[{i+1}] Original : {df["review"].iloc[i][:80]}...')
    print(f'    Cleaned  : {df["clean_review"].iloc[i]}')

---
## 🔢 Step 3: Feature Engineering

We convert cleaned text into numerical vectors using **Bag of Words** and **TF-IDF**.

In [ ]:
# ── Encode labels: positive=1, negative=0 ──
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

X = df['clean_review']
y = df['label']

# ── Train/Test Split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {len(X_train)}')
print(f'Test size  : {len(X_test)}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')
print(f'Test  class balance: {y_test.value_counts().to_dict()}')

In [ ]:
# ── 3.1 Bag of Words (BoW) ──
# Counts how many times each word appears in each document.
bow_vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_bow    = bow_vectorizer.fit_transform(X_train)
X_test_bow     = bow_vectorizer.transform(X_test)

print('✅ Bag of Words Vectorization:')
print(f'   Vocabulary size : {len(bow_vectorizer.vocabulary_)}')
print(f'   Train matrix    : {X_train_bow.shape}')
print(f'   Test  matrix    : {X_test_bow.shape}')
print(f'   Sparsity        : {100 * (1 - X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1])):.1f}%')

In [ ]:
# ── 3.2 TF-IDF ──
# Weighs words by how unique they are across all documents.
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf    = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf     = tfidf_vectorizer.transform(X_test)

print('✅ TF-IDF Vectorization:')
print(f'   Vocabulary size : {len(tfidf_vectorizer.vocabulary_)}')
print(f'   Train matrix    : {X_train_tfidf.shape}')
print(f'   Test  matrix    : {X_test_tfidf.shape}')

# Show top TF-IDF terms
feature_names = tfidf_vectorizer.get_feature_names_out()
mean_tfidf    = X_train_tfidf.mean(axis=0).A1
top_indices   = mean_tfidf.argsort()[-10:][::-1]
print(f'\nTop 10 TF-IDF terms: {[feature_names[i] for i in top_indices]}')

---
## 🤖 Step 4: Model Building & Training

We train **4 models** on both BoW and TF-IDF features.

In [ ]:
# ── Define all models ──
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes'         : MultinomialNB(alpha=0.5),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

# ── Storage for all results ──
results = []

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, vectorizer_name):
    """
    Train a model, evaluate it, and return a results dictionary.
    Prints a detailed classification report.
    """
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    acc  = accuracy_score(y_te,  y_pred)
    prec = precision_score(y_te, y_pred, average='weighted')
    rec  = recall_score(y_te,    y_pred, average='weighted')
    f1   = f1_score(y_te,        y_pred, average='weighted')

    print(f'\n{"-"*55}')
    print(f'Model      : {name}')
    print(f'Vectorizer : {vectorizer_name}')
    print(f'Accuracy   : {acc:.4f}')
    print(f'Precision  : {prec:.4f}')
    print(f'Recall     : {rec:.4f}')
    print(f'F1 Score   : {f1:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(y_te, y_pred, target_names=['Negative', 'Positive']))

    return {
        'Model'      : name,
        'Vectorizer' : vectorizer_name,
        'Accuracy'   : round(acc,  4),
        'Precision'  : round(prec, 4),
        'Recall'     : round(rec,  4),
        'F1 Score'   : round(f1,   4),
        'trained'    : model,
        'y_pred'     : y_pred,
    }


print('✅ Models and evaluation function ready!')
print(f'   Models to train : {list(models.keys())}')
print(f'   Vectorizers     : BoW, TF-IDF')
print(f'   Total runs      : {len(models) * 2}')

In [ ]:
# ── Train all models on BoW features ──
print('=' * 55)
print('TRAINING ON BAG OF WORDS')
print('=' * 55)

for name, model in models.items():
    res = evaluate_model(
        name, model,
        X_train_bow, X_test_bow,
        y_train, y_test,
        'BoW'
    )
    results.append(res)

In [ ]:
# ── Train all models on TF-IDF features ──
print('=' * 55)
print('TRAINING ON TF-IDF')
print('=' * 55)

# Re-initialise models (fresh instances) for TF-IDF run
models_tfidf = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes'         : MultinomialNB(alpha=0.5),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

for name, model in models_tfidf.items():
    res = evaluate_model(
        name, model,
        X_train_tfidf, X_test_tfidf,
        y_train, y_test,
        'TF-IDF'
    )
    results.append(res)

---
## 📊 Step 5: Model Evaluation & Comparison

In [ ]:
# ── Build comparison DataFrame ──
results_df = pd.DataFrame([{
    'Model'      : r['Model'],
    'Vectorizer' : r['Vectorizer'],
    'Accuracy'   : r['Accuracy'],
    'Precision'  : r['Precision'],
    'Recall'     : r['Recall'],
    'F1 Score'   : r['F1 Score'],
} for r in results])

results_df = results_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)

print('=' * 70)
print('FULL MODEL COMPARISON TABLE')
print('=' * 70)
print(results_df.to_string(index=False))
print()

best = results_df.iloc[0]
print(f'🏆 Best Model : {best["Model"]} with {best["Vectorizer"]}')
print(f'   F1 Score   : {best["F1 Score"]}')
print(f'   Accuracy   : {best["Accuracy"]}')

In [ ]:
# ── Visual 1: F1 Score comparison bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, vec in zip(axes, ['BoW', 'TF-IDF']):
    sub = results_df[results_df['Vectorizer'] == vec].copy()
    colors = ['#4CAF50' if v == sub['F1 Score'].max() else '#2196F3'
              for v in sub['F1 Score']]
    bars = ax.bar(sub['Model'], sub['F1 Score'], color=colors,
                  edgecolor='white', linewidth=1.5)
    ax.set_title(f'F1 Score – {vec}', fontsize=13, fontweight='bold')
    ax.set_ylabel('F1 Score')
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(sub['Model'], rotation=15, ha='right')
    for bar, val in zip(bars, sub['F1 Score']):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Model F1 Score Comparison – BoW vs TF-IDF',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Visual 2: All metrics heatmap ──
pivot = results_df.set_index(['Model', 'Vectorizer'])[['Accuracy', 'Precision', 'Recall', 'F1 Score']]

plt.figure(figsize=(10, 6))
sns.heatmap(
    pivot,
    annot=True, fmt='.3f', cmap='YlGn',
    linewidths=0.5, linecolor='white',
    vmin=0.5, vmax=1.0
)
plt.title('All Metrics Heatmap – Model × Vectorizer', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visual 3: Confusion Matrices for best models (per vectorizer) ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vec in zip(axes, ['BoW', 'TF-IDF']):
    # Find best result for this vectorizer
    best_res = max(
        [r for r in results if r['Vectorizer'] == vec],
        key=lambda x: x['F1 Score']
    )
    cm = confusion_matrix(y_test, best_res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Negative', 'Positive'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'Confusion Matrix\n{best_res["Model"]} ({vec})',
                 fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Visual 4: Grouped bar chart – all metrics ──
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

for ax, vec in zip(axes, ['BoW', 'TF-IDF']):
    sub = results_df[results_df['Vectorizer'] == vec]
    x   = np.arange(len(sub))
    w   = 0.2
    colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']

    for i, (metric, color) in enumerate(zip(metrics, colors)):
        ax.bar(x + i * w, sub[metric], w, label=metric,
               color=color, alpha=0.85, edgecolor='white')

    ax.set_xticks(x + w * 1.5)
    ax.set_xticklabels(sub['Model'], rotation=10)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'All Metrics – {vec}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score')
    ax.legend(loc='lower right')
    ax.axhline(0.9, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

plt.suptitle('Model Evaluation – All Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 💡 Step 6: Insights, Comparison & Conclusions

In [ ]:
# ── Top features from Logistic Regression + TF-IDF ──
# (Most informative words for each class)

# Get the LR + TF-IDF model from results
lr_tfidf_res = next(r for r in results
                    if r['Model'] == 'Logistic Regression' and r['Vectorizer'] == 'TF-IDF')
lr_model = lr_tfidf_res['trained']

feature_names = tfidf_vectorizer.get_feature_names_out()
coef          = lr_model.coef_[0]

top_pos_idx = coef.argsort()[-15:][::-1]
top_neg_idx = coef.argsort()[:15]

print('🟢 Top 15 words STRONGLY associated with POSITIVE sentiment:')
for i, idx in enumerate(top_pos_idx, 1):
    print(f'   {i:2}. {feature_names[idx]:<25} (coef: {coef[idx]:+.4f})')

print()
print('🔴 Top 15 words STRONGLY associated with NEGATIVE sentiment:')
for i, idx in enumerate(top_neg_idx, 1):
    print(f'   {i:2}. {feature_names[idx]:<25} (coef: {coef[idx]:+.4f})')

In [ ]:
# ── Visual: Top positive & negative words ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Positive words
pos_words  = [feature_names[i] for i in top_pos_idx[:10]]
pos_scores = [coef[i] for i in top_pos_idx[:10]]
axes[0].barh(pos_words[::-1], pos_scores[::-1], color='#4CAF50', edgecolor='white')
axes[0].set_title('Top Words → Positive Sentiment', fontweight='bold')
axes[0].set_xlabel('Coefficient (LR + TF-IDF)')

# Negative words
neg_words  = [feature_names[i] for i in top_neg_idx[:10]]
neg_scores = [coef[i] for i in top_neg_idx[:10]]
axes[1].barh(neg_words[::-1], neg_scores[::-1], color='#F44336', edgecolor='white')
axes[1].set_title('Top Words → Negative Sentiment', fontweight='bold')
axes[1].set_xlabel('Coefficient (LR + TF-IDF)')

plt.suptitle('Most Informative Features – Logistic Regression + TF-IDF',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Live prediction demo ──
def predict_sentiment(review_text):
    """
    Predict sentiment of a new review using best model
    (Logistic Regression + TF-IDF).
    """
    cleaned   = preprocess(review_text)
    vectorized = tfidf_vectorizer.transform([cleaned])
    pred       = lr_model.predict(vectorized)[0]
    prob       = lr_model.predict_proba(vectorized)[0]
    label      = 'POSITIVE 😊' if pred == 1 else 'NEGATIVE 😞'
    confidence = max(prob) * 100

    print(f'Review    : {review_text[:80]}')
    print(f'Cleaned   : {cleaned[:80]}')
    print(f'Sentiment : {label}')
    print(f'Confidence: {confidence:.1f}%')
    print('-' * 55)

print('=' * 55)
print('LIVE SENTIMENT PREDICTION DEMO')
print('=' * 55)

test_reviews = [
    "This movie was absolutely fantastic! One of the best films I have ever seen.",
    "Terrible waste of time. The plot made no sense at all.",
    "Not bad but not great either. Average performance by the cast.",
    "I loved every single minute of this masterpiece!",
    "The worst film I have ever seen. Complete rubbish.",
]

for rev in test_reviews:
    print()
    predict_sentiment(rev)

In [ ]:
# ── Final Summary ──
print('=' * 70)
print('FINAL SUMMARY & INSIGHTS')
print('=' * 70)

print("""
1. BEST PREPROCESSING STEPS
   - Lowercasing + punctuation removal standardise text effectively.
   - Preserving negation words ('not', 'never') is critical for sentiment.
   - Lemmatization outperforms stemming as it always produces valid words.
   - Removing URLs, emojis and repeated characters reduces noise significantly.

2. BEST VECTORIZATION
   - TF-IDF outperforms Bag of Words across all models.
   - TF-IDF weights rare but meaningful words higher (e.g., 'masterpiece').
   - BoW gives equal weight to all words, making it noisier for sentiment tasks.
   - Bigrams (ngram_range=(1,2)) capture important phrases like 'not good'.

3. BEST MODEL
   - Logistic Regression + TF-IDF achieves the best F1 Score.
   - It is fast, interpretable, and generalises well on text data.
   - Naive Bayes is close behind and trains much faster (good baseline).
   - Decision Tree tends to overfit; lower F1 compared to LR.
   - Random Forest improves over Decision Tree but is much slower.

4. TRADE-OFFS
   Model              Speed       Accuracy   Interpretability
   Logistic Regression  Fast        High       High
   Naive Bayes          Very Fast   Good       Medium
   Decision Tree        Fast        Low        High
   Random Forest        Slow        Good       Low

5. KEY TAKEAWAY
   A well-preprocessed dataset with TF-IDF features + Logistic Regression
   is a strong, fast baseline for binary sentiment analysis tasks.
""")

print('=' * 70)
print(f'Best Result  → {results_df.iloc[0]["Model"]} + {results_df.iloc[0]["Vectorizer"]}')
print(f'F1 Score     → {results_df.iloc[0]["F1 Score"]}')
print(f'Accuracy     → {results_df.iloc[0]["Accuracy"]}')
print('=' * 70)